You will design and implement a pipeline that, for each (case_id, condition):
1. Builds a prompt and queries the model K = 50 times with temperature=1.0 passed explicitly on every call (do not omit the parameter and rely on the API default). The exact model(s)
you call are determined by your axis assignment (see the table in Your Assignment below).
2. Parses each response to extract a clean answer (a single letter "A"-"D" for GPQA, a Python
int in [0, 999] for AIME). Responses you cannot parse should collapse to the literal string
"_UNPARSEABLE_" — the catch-all bucket for malformed model output. _UNPARSEABLE_ entries count as wrong when computing accuracy A.
3. Computes per-case metrics — consistency C, accuracy A, risk-adjusted accuracy 𝑅𝛼, and
the Wilson lower bound R_wilson (formulas in The Data → Metrics below).
4. Validates and serializes the results into a single {PennKey}_results.json (for example,
dkaihua_results.json) that satisfie

In [ ]:
#Preliminaries - Install Packages
from openai import OpenAI
import scipy, jsonschema, json
import os
from dotenv import load_dotenv

length_id_batch = 25

#Hidden OpenAI API Key 
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("API Key not found! Ensure you have a .env file with OPENAI_API_KEY defined.")

client = OpenAI(api_key=api_key)

#Pull in AIME and GPQA Diamond
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)

# Load the GPQA pool
with open('gpqa_pool.json', 'r') as f:
    gpqa_data = json.load(f)

#load case_ids assigned to me
with open('aime_pool.json', 'r') as f:
    aime_data = json.load(f)

# Load the GPQA pool
with open('asudit.json', 'r') as f:
    asudit_data = json.load(f)

# Verify the data loaded correctly
print(f"Loaded {len(aime_data)} items from AIME")
print(f"Loaded {len(gpqa_data)} items from GPQA")
print(f"Loaded {len(asudit_data)} items from asudit")

#split case_ids
gpqa_id = asudit_data["case_ids"][:length_id_batch]
aime_id = asudit_data["case_ids"][length_id_batch:]

#print(gpqa_id)
#print(aime_id)

Loaded 196 items from AIME
Loaded 198 items from GPQA
Loaded 4 items from asudit
['gpqa_d_157', 'gpqa_d_151', 'gpqa_d_032', 'gpqa_d_096', 'gpqa_d_101', 'gpqa_d_015', 'gpqa_d_072', 'gpqa_d_002', 'gpqa_d_113', 'gpqa_d_075', 'gpqa_d_156', 'gpqa_d_016', 'gpqa_d_165', 'gpqa_d_190', 'gpqa_d_028', 'gpqa_d_120', 'gpqa_d_171', 'gpqa_d_044', 'gpqa_d_159', 'gpqa_d_021', 'gpqa_d_180', 'gpqa_d_074', 'gpqa_d_121', 'gpqa_d_189', 'gpqa_d_134']
['aime_160', 'aime_076', 'aime_161', 'aime_019', 'aime_192', 'aime_005', 'aime_167', 'aime_112', 'aime_096', 'aime_012', 'aime_087', 'aime_032', 'aime_009', 'aime_169', 'aime_179', 'aime_027', 'aime_141', 'aime_138', 'aime_049', 'aime_050', 'aime_033', 'aime_026', 'aime_149', 'aime_117', 'aime_157']


Now We Need to Call OpenAI API and Enter some Prompts

In [ ]:
#Axis C mid-vs-frontier: gpt-4.1-mini (zero-shot) [Condition A] vs gpt-4.1 (zero-shot) [Condition B]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "system", "content": "You are a helpful assistant."},
              {"role": "user","content": "What is an AI agent in one sentence?"},
              ],
max_tokens=1000,
)

response = client.chat.completions.create(
    model="gpt-4.1",
    messages=[{"role": "system", "content": "You are a helpful assistant."},
              {"role": "user","content": "What is an AI agent in one sentence?"},
              ],
max_tokens=1000,
)